# Dispute + Coalition Training — Round 2 Retry

This notebook trains the multi-agent **dispute** and **coalition** tasks that did not converge in the main training run. Applies four mitigations: more rollouts per batch, higher learning rate, lower KL penalty, longer training.

Run sequentially. Cell N depends on Cell N-1. Both training runs take approximately 60–90 minutes each on a Colab T4.

**Required Colab secrets:** `HF_TOKEN`, `WANDB_API_KEY`.


In [ ]:
!pip install -q unsloth trl openenv-core wandb requests pydantic


In [ ]:
import os, json, time, re, statistics, gc
import requests
import torch

# Pull secrets from Colab if present.
try:
    from google.colab import userdata
    for _name in ("HF_TOKEN", "WANDB_API_KEY"):
        try:
            _v = userdata.get(_name)
            if _v:
                os.environ[_name] = _v
        except Exception:
            pass
except ImportError:
    pass

if not os.environ.get("WANDB_API_KEY"):
    os.environ.setdefault("WANDB_MODE", "offline")
    print("[setup] WANDB_API_KEY not set; using WANDB_MODE=offline.")

import wandb
try:
    wandb.login(anonymous="allow")
except Exception as e:
    print(f"[setup] wandb.login skipped: {e}")

print("Imports OK.")


In [ ]:
SPACE_URL = "https://ren9087-rf-spectrum-env-v2.hf.space"

def wake_space(url=SPACE_URL, attempts=15, delay=10):
    for i in range(1, attempts + 1):
        try:
            r = requests.get(f"{url}/health", timeout=20)
            if r.status_code == 200:
                print(f"Space awake on attempt {i}: {r.text[:120]}")
                return True
        except requests.RequestException as e:
            print(f"[wake] attempt {i}/{attempts} failed: {e}")
        time.sleep(delay)
    raise RuntimeError(f"Space at {url} did not wake after {attempts} attempts")

wake_space()


In [ ]:
# Dispute sanity check
r = requests.post(f"{SPACE_URL}/reset",
                  json={"seed": 42, "task": "dispute"})
assert r.status_code == 200, f"dispute reset failed: {r.text}"
test_action = {"dispute_choice": "negotiate",
               "justification": "test"}
r = requests.post(f"{SPACE_URL}/step",
                  json={"action": test_action})
assert r.status_code == 200, f"dispute step failed: {r.text}"
print("Dispute task OK")

# Coalition sanity check
r = requests.post(f"{SPACE_URL}/reset",
                  json={"seed": 42, "task": "coalition"})
assert r.status_code == 200, f"coalition reset failed: {r.text}"
test_action = {"cooperation_flag": "cooperate",
               "justification": "test"}
r = requests.post(f"{SPACE_URL}/step",
                  json={"action": test_action})
assert r.status_code == 200, f"coalition step failed: {r.text}"
print("Coalition task OK")


In [ ]:
from unsloth import FastLanguageModel

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL_ID,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    fast_inference=True,
    gpu_memory_utilization=0.5,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=16,
    use_gradient_checkpointing="unsloth",
)

torch.cuda.synchronize()
print(f"Model loaded. VRAM: {torch.cuda.memory_allocated()/1024**3:.2f} GB")


## Mitigation Strategy

These tasks failed to learn with the auction notebook's settings. Four changes applied:

1. **`num_generations=8`** — twice the auction notebook's 4. Small action spaces (3 choices for coalition, 4 for dispute) need more rollouts per batch to provide ranking diversity.
2. **`learning_rate=2e-5`** — 4× the auction notebook's `5e-6`. Small gradient signals on sparse rewards need stronger updates.
3. **`beta=0.01`** — explicit low KL coefficient. Default TRL KL penalty is too strong for tasks where the policy needs to move significantly to find better strategies.
4. **`max_steps=200`** — 4× the original 50. Short runs gave GRPO no time to find signal in stochastic-reward settings.


In [ ]:
DISPUTE_SYSTEM_PROMPT = (
    "You are a player in a one-shot spectrum dispute resolution game with a regulator referee. "
    "Pick exactly ONE of these four actions:\n\n"
    "- CONCEDE: yield the contested band; lowest payoff but no risk\n"
    "- NEGOTIATE: propose a split; medium payoff, robust to opponent type\n"
    "- ESCALATE: refuse to yield; high payoff vs conservative opponents, mutual loss vs "
    "aggressive opponents (regulator may issue a VIOLATION)\n"
    "- AUDIT: trigger a regulator audit; medium payoff, useful when you suspect rule-breaking\n\n"
    "Observe the competitor's behavior and infer their type before choosing. "
    "Think briefly, then output ONLY a JSON object:\n\n"
    '{"dispute_choice": "negotiate", "justification": "<one short sentence>"}\n\n'
    'The dispute_choice value MUST be one of: "concede", "negotiate", "escalate", "audit".'
)

COALITION_SYSTEM_PROMPT = (
    "You are a player in an iterated coalition game over a shared spectrum resource, "
    "with a reputation-tracking regulator. Pick exactly ONE of these three actions each round:\n\n"
    "- COOPERATE: share the resource; raises your reputation (+0.05)\n"
    "- DEFECT: take more than your share; lowers reputation (-0.10) and risks a regulator "
    "WARNING when reputation is high\n"
    "- ABSTAIN: skip this round; reputation unchanged\n\n"
    "Cooperation builds reputation that compounds across episodes. Defection wins short-term "
    "but attracts regulator scrutiny when your reputation is already high. "
    "Output ONLY a JSON object:\n\n"
    '{"cooperation_flag": "cooperate", "justification": "<one short sentence>"}\n\n'
    'The cooperation_flag value MUST be one of: "cooperate", "defect", "abstain".'
)

print("Prompts ready.")


In [ ]:
VALID_DISPUTE = {"concede", "negotiate", "escalate", "audit"}

def _build_user_prompt(observation: dict, task: str) -> str:
    """Render an observation dict into a compact natural-language user turn."""
    fields = []
    for k in ("round_index", "total_rounds", "remaining_budget",
              "reputation_score", "competitor_bid_history",
              "oversight_events"):
        v = observation.get(k)
        if v is not None and v != [] and v != {}:
            fields.append(f"{k}: {json.dumps(v, default=str)[:280]}")
    return f"Task: {task}\n" + "\n".join(fields)

def _parse_dispute_action(text: str) -> dict:
    """Extract a dispute_choice/justification dict from raw model text.
    Defaults to all-None on parse failure (env treats it as a no-op)."""
    try:
        m = re.search(r"\{[^{}]*\}", text, flags=re.DOTALL)
        if not m:
            return {"dispute_choice": None, "justification": ""}
        obj = json.loads(m.group(0))
        choice = str(obj.get("dispute_choice", "")).lower().strip()
        if choice not in VALID_DISPUTE:
            choice = None
        return {
            "dispute_choice": choice,
            "justification": str(obj.get("justification", ""))[:500],
        }
    except Exception:
        return {"dispute_choice": None, "justification": ""}

def dispute_rollout_func(prompts, **kwargs):
    """GRPO rollout for the dispute task.

    For each prompt slot, reset the env on a fresh seed, generate a
    completion via vLLM, parse the action, step the env once, and return
    the structures TRL expects (token ids, logprobs) plus per-component
    env_reward_<name> kwargs that the reward funcs read back.
    """
    from vllm import SamplingParams
    sampling_params = SamplingParams(temperature=0.9, top_p=0.95, max_tokens=256)

    seeds = [int(time.time() * 1e6) % 200 + i for i in range(len(prompts))]

    completion_texts = []
    revenues, interferences, compliances, justifications = [], [], [], []

    for prompt_text, seed in zip(prompts, seeds):
        try:
            r = requests.post(
                f"{SPACE_URL}/reset",
                json={"seed": int(seed), "task": "dispute"},
                timeout=30,
            )
            obs = r.json().get("observation", {}) if r.ok else {}
        except Exception:
            obs = {}

        msgs = [
            {"role": "system", "content": DISPUTE_SYSTEM_PROMPT},
            {"role": "user", "content": _build_user_prompt(obs, "dispute")},
        ]
        formatted = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True,
        )

        out = model.fast_generate([formatted], sampling_params=sampling_params)
        completion_text = out[0].outputs[0].text
        completion_texts.append(completion_text)

        action = _parse_dispute_action(completion_text)
        try:
            r = requests.post(
                f"{SPACE_URL}/step",
                json={"action": action},
                timeout=30,
            )
            step_obs = r.json().get("observation", {}) if r.ok else {}
            comps = (step_obs.get("metadata") or {}).get("reward_components") or {}
        except Exception:
            comps = {}

        revenues.append(float(comps.get("revenue", 0.0)))
        interferences.append(float(comps.get("interference", 0.0)))
        compliances.append(float(comps.get("compliance", 0.0)))
        justifications.append(float(comps.get("justification", 0.0)))

    prompt_ids = [tokenizer.encode(p, add_special_tokens=False) for p in prompts]
    completion_ids = [tokenizer.encode(c, add_special_tokens=False) for c in completion_texts]
    logprobs = [[0.0] * len(c) for c in completion_ids]

    return {
        "prompt_ids": prompt_ids,
        "completion_ids": completion_ids,
        "logprobs": logprobs,
        "env_reward_revenue": revenues,
        "env_reward_interference": interferences,
        "env_reward_compliance": compliances,
        "env_reward_justification": justifications,
    }


In [ ]:
def extract_revenue(completions, **kwargs):
    return kwargs.get("env_reward_revenue", [0.0] * len(completions))

def extract_interference(completions, **kwargs):
    return kwargs.get("env_reward_interference", [0.0] * len(completions))

def extract_compliance(completions, **kwargs):
    return kwargs.get("env_reward_compliance", [0.0] * len(completions))

def extract_justification(completions, **kwargs):
    return kwargs.get("env_reward_justification", [0.0] * len(completions))


In [ ]:
from trl import GRPOConfig

dispute_config = GRPOConfig(
    output_dir="rf-spectrum-dispute-trained",
    learning_rate=2e-5,           # 4x auction's value
    num_generations=8,            # 2x auction's value
    beta=0.01,                    # low KL penalty
    max_prompt_length=1024,
    max_completion_length=256,
    num_train_epochs=1,
    max_steps=200,                # 4x auction's smoke value
    use_vllm=True,
    vllm_mode="colocate",
    report_to="wandb" if os.environ.get("WANDB_API_KEY") else "none",
    logging_steps=2,
    save_steps=50,
    bf16=True,
    run_name="rf-spectrum-dispute-mitigated-v1",
    remove_unused_columns=False,
)


In [ ]:
from trl import GRPOTrainer
from datasets import Dataset

# Build a minimal training dataset — one entry per intended training step.
# Actual prompt content is regenerated inside the rollout_func from fresh
# env resets, so the dataset just provides iteration scaffolding.
_dispute_rows = [{"prompt": "dispute", "seed": s} for s in range(200)]
dispute_train_dataset = Dataset.from_list(_dispute_rows)

dispute_trainer = GRPOTrainer(
    model=model,
    reward_funcs=[extract_revenue, extract_interference,
                  extract_compliance, extract_justification],
    args=dispute_config,
    train_dataset=dispute_train_dataset,
    rollout_func=dispute_rollout_func,
)
dispute_trainer.train()


In [ ]:
model.save_pretrained_merged(
    "rf-spectrum-dispute-trained",
    tokenizer,
    save_method="lora",
)
print("Dispute checkpoint saved.")


## Coalition training

Coalition is the structurally hardest task: cross-episode reputation effects mean that the “right” action depends on a horizon longer than a single episode. Same mitigations applied. Expect slower convergence than dispute.


In [ ]:
VALID_COALITION = {"cooperate", "defect", "abstain"}

def _parse_coalition_action(text: str) -> dict:
    try:
        m = re.search(r"\{[^{}]*\}", text, flags=re.DOTALL)
        if not m:
            return {"cooperation_flag": None, "justification": ""}
        obj = json.loads(m.group(0))
        flag = str(obj.get("cooperation_flag", "")).lower().strip()
        if flag not in VALID_COALITION:
            flag = None
        return {
            "cooperation_flag": flag,
            "justification": str(obj.get("justification", ""))[:500],
        }
    except Exception:
        return {"cooperation_flag": None, "justification": ""}

def coalition_rollout_func(prompts, **kwargs):
    from vllm import SamplingParams
    sampling_params = SamplingParams(temperature=0.9, top_p=0.95, max_tokens=256)

    seeds = [int(time.time() * 1e6) % 200 + i for i in range(len(prompts))]

    completion_texts = []
    revenues, interferences, compliances, justifications = [], [], [], []

    for prompt_text, seed in zip(prompts, seeds):
        try:
            r = requests.post(
                f"{SPACE_URL}/reset",
                json={"seed": int(seed), "task": "coalition"},
                timeout=30,
            )
            obs = r.json().get("observation", {}) if r.ok else {}
        except Exception:
            obs = {}

        msgs = [
            {"role": "system", "content": COALITION_SYSTEM_PROMPT},
            {"role": "user", "content": _build_user_prompt(obs, "coalition")},
        ]
        formatted = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True,
        )

        out = model.fast_generate([formatted], sampling_params=sampling_params)
        completion_text = out[0].outputs[0].text
        completion_texts.append(completion_text)

        action = _parse_coalition_action(completion_text)
        try:
            r = requests.post(
                f"{SPACE_URL}/step",
                json={"action": action},
                timeout=30,
            )
            step_obs = r.json().get("observation", {}) if r.ok else {}
            comps = (step_obs.get("metadata") or {}).get("reward_components") or {}
        except Exception:
            comps = {}

        revenues.append(float(comps.get("revenue", 0.0)))
        interferences.append(float(comps.get("interference", 0.0)))
        compliances.append(float(comps.get("compliance", 0.0)))
        justifications.append(float(comps.get("justification", 0.0)))

    prompt_ids = [tokenizer.encode(p, add_special_tokens=False) for p in prompts]
    completion_ids = [tokenizer.encode(c, add_special_tokens=False) for c in completion_texts]
    logprobs = [[0.0] * len(c) for c in completion_ids]

    return {
        "prompt_ids": prompt_ids,
        "completion_ids": completion_ids,
        "logprobs": logprobs,
        "env_reward_revenue": revenues,
        "env_reward_interference": interferences,
        "env_reward_compliance": compliances,
        "env_reward_justification": justifications,
    }


In [ ]:
coalition_config = GRPOConfig(
    output_dir="rf-spectrum-coalition-trained",
    learning_rate=2e-5,
    num_generations=8,
    beta=0.01,
    max_prompt_length=1024,
    max_completion_length=256,
    num_train_epochs=1,
    max_steps=200,
    use_vllm=True,
    vllm_mode="colocate",
    report_to="wandb" if os.environ.get("WANDB_API_KEY") else "none",
    logging_steps=2,
    save_steps=50,
    bf16=True,
    run_name="rf-spectrum-coalition-mitigated-v1",
    remove_unused_columns=False,
)


In [ ]:
_coalition_rows = [{"prompt": "coalition", "seed": s} for s in range(200)]
coalition_train_dataset = Dataset.from_list(_coalition_rows)

coalition_trainer = GRPOTrainer(
    model=model,
    reward_funcs=[extract_revenue, extract_interference,
                  extract_compliance, extract_justification],
    args=coalition_config,
    train_dataset=coalition_train_dataset,
    rollout_func=coalition_rollout_func,
)
coalition_trainer.train()


In [ ]:
model.save_pretrained_merged(
    "rf-spectrum-coalition-trained",
    tokenizer,
    save_method="lora",
)
print("Coalition checkpoint saved.")


## What to do with results

- **If reward curves visibly rise:** include in submission, mention as secondary results to auction.
- **If reward curves stay flat:** the tasks are confirmed harder than auction in the current training budget. Document this honestly in the README/blog as “additional implemented tasks; convergence requires curriculum learning or longer horizons we did not have compute to apply.” Do not show flat curves as if they were training successes.

Either outcome is informative. Honest negative results are better than fabricated positive results.


In [ ]:
from vllm import SamplingParams

EVAL_SEEDS = list(range(200, 230))

def _generate_action_text(messages):
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    out = model.fast_generate(
        [formatted],
        sampling_params=SamplingParams(temperature=0.0, max_tokens=256),
    )
    return out[0].outputs[0].text

def _play_episode(task: str, seed: int) -> float:
    """Play one full episode using the current in-memory policy.
    Returns mean reward across the episode's steps."""
    if task == "dispute":
        system_prompt = DISPUTE_SYSTEM_PROMPT
        parser = _parse_dispute_action
    elif task == "coalition":
        system_prompt = COALITION_SYSTEM_PROMPT
        parser = _parse_coalition_action
    else:
        raise ValueError(task)

    r = requests.post(
        f"{SPACE_URL}/reset",
        json={"seed": int(seed), "task": task},
        timeout=30,
    )
    if not r.ok:
        return 0.0
    obs = r.json().get("observation", {})

    rewards = []
    done = False
    safety_cap = 12  # all Round 2 tasks <= 6 rounds
    while not done and safety_cap > 0:
        msgs = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": _build_user_prompt(obs, task)},
        ]
        text = _generate_action_text(msgs)
        action = parser(text)
        r = requests.post(f"{SPACE_URL}/step", json={"action": action}, timeout=30)
        if not r.ok:
            break
        body = r.json()
        obs = body.get("observation", {})
        rewards.append(float(body.get("reward", 0.0)))
        done = bool(body.get("done", False))
        safety_cap -= 1
    return statistics.fmean(rewards) if rewards else 0.0

def evaluate_task(task_name: str, seeds=EVAL_SEEDS) -> dict:
    """Run held-out evaluation and return summary stats."""
    means = [_play_episode(task_name, s) for s in seeds]
    return {
        "task": task_name,
        "n_seeds": len(seeds),
        "mean_reward": statistics.fmean(means),
        "stdev_reward": statistics.pstdev(means) if len(means) > 1 else 0.0,
        "per_seed": means,
    }

# Compare against baselines.json values:
#   dispute baseline:   0.14
#   coalition baseline: 0.1175
dispute_eval   = evaluate_task("dispute")
coalition_eval = evaluate_task("coalition")

print("=" * 64)
print(f"DISPUTE   trained mean: {dispute_eval['mean_reward']:.4f}  (baseline 0.1400)")
print(f"COALITION trained mean: {coalition_eval['mean_reward']:.4f}  (baseline 0.1175)")
print("=" * 64)

with open("training/dispute_coalition_eval.json", "w") as f:
    json.dump({"dispute": dispute_eval, "coalition": coalition_eval}, f, indent=2)
print("Saved: training/dispute_coalition_eval.json")
